# Mastering LLM Deployment
## Day 1 · Lab 5 - Capstone: Stacking the Optimizations

**Duration:** ~90 minutes  ·  **Runtime:** T4 GPU  ·  **Prerequisites:** Labs 1–4

---

### The question this lab answers

Labs 2, 3 and 4 each proved one lever works in isolation. That is not the same as proving they work **together**. A model that has already had two-thirds of its layers removed may have far less redundancy left for pruning to exploit; a model whose weights were reshaped by distillation may quantize differently.

So we do what the case study did: apply the levers in order, **measuring after every step**, and see whether the gains compound.

We follow the order the case study used:

1. **Smaller model** - distil BERT-base into a 4-layer student.
2. **Smaller model again** - structured feed-forward pruning on the student.
3. **Smaller weights** - int8 quantization of what remains.
4. **Smaller inputs** - drop fixed-length padding for dynamic shapes.

Then we measure on **CPU**, because that is the substrate Day 2 deploys to and it is where these optimizations pay off most.

### What you produce

A single deployable artifact - a `SavedModel` with a dynamic-shape serving signature, its tokenizer, and a deployment manifest - that Day 2 packages into a Docker image and runs on AWS ECS behind TensorFlow Serving and a Flask API.

### Learning outcomes

- Compose distillation, pruning and quantization and quantify the compounding.
- Measure on the target hardware class, not the training hardware.
- Quantify the cost of fixed-length padding and design a dynamic-shape serving signature.
- Export and validate a production artifact, and write the handoff document a deployment engineer needs.

**Expected GPU time: 10–15 minutes.**

---
## 0. Environment setup

In [ ]:
%pip install -q "transformers>=4.40,<5" "datasets>=2.19,<4" "tf-keras>=2.16" "tensorflow-model-optimization>=0.8.0" "scikit-learn" "pandas" "matplotlib"
print('dependencies installed')

In [ ]:
import os, sys
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
if "tensorflow" in sys.modules:
    print("TensorFlow already imported -> Runtime > Restart session and re-run from the top.")

import tensorflow as tf, numpy as np, pandas as pd, json, time
# tf.keras is a lazy loader and does not re-export __version__.
try:
    import tf_keras as _keras_pkg
except ImportError:
    import keras as _keras_pkg
_keras_impl = tf.keras.Model.__module__
print("TF", tf.__version__, "| Keras", _keras_pkg.__version__, "|", _keras_impl)
assert _keras_pkg.__version__.startswith("2.") and "tf_keras" in _keras_impl, (
    "Keras 2 is not active. Install tf-keras, set TF_USE_LEGACY_KERAS=1 before "
    "importing tensorflow, then Runtime > Restart session.")
tf.keras.utils.set_random_seed(42)

In [ ]:
USE_DRIVE = True
ROOT = "/content/llm-deploy-labs"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = "/content/drive/MyDrive/llm-deploy-labs"
    except Exception as e:
        print("Drive unavailable:", e)
os.environ["LLMDEPLOY_ROOT"] = ROOT
for sub in ("models", "reports", "data"):
    os.makedirs(os.path.join(ROOT, sub), exist_ok=True)
print("Artifact root:", ROOT)

In [ ]:
LABKIT_SRC = r'''
"""
labkit.py - shared utilities for the "Mastering LLM Deployment" hands-on labs.

Everything the labs need in common lives here so that each notebook measures
the same things in the same way:

  * artifact + ledger management (results survive across notebooks via Drive)
  * a model "size on disk" and parameter/sparsity accounting
  * a latency/throughput benchmark harness with warm-up and percentiles
  * a minimal, explicit GradientTape training loop (works for HF TF models,
    plain Keras models, distillation losses and masked/pruned training alike)
"""

import os

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")

import json
import shutil
import time
from pathlib import Path

import numpy as np
import tensorflow as tf

# --------------------------------------------------------------------------
# 1. Artifact root
# --------------------------------------------------------------------------

_ROOT = Path(os.environ.get("LLMDEPLOY_ROOT", "/content/llm-deploy-labs"))


def set_root(path):
    """Point the lab kit at a persistent directory (ideally on Google Drive)."""
    global _ROOT
    _ROOT = Path(path)
    for sub in ("models", "reports", "data"):
        (_ROOT / sub).mkdir(parents=True, exist_ok=True)
    os.environ["LLMDEPLOY_ROOT"] = str(_ROOT)
    return _ROOT


def root():
    return _ROOT


def model_dir(name, clean=False):
    """Return (and create) a directory under <root>/models/<name>."""
    d = _ROOT / "models" / name
    if clean and d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
    return d


# --------------------------------------------------------------------------
# 2. Size and parameter accounting
# --------------------------------------------------------------------------


def size_mb(path):
    """Size of a file or, recursively, of a directory - in MB."""
    p = Path(path)
    if p.is_file():
        return p.stat().st_size / 1e6
    total = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
    return total / 1e6


def count_params(model):
    """Total trainable parameter count."""
    return int(sum(int(np.prod(v.shape)) for v in model.trainable_variables))


def weight_sparsity(model, kinds=("kernel", "weight", "embeddings")):
    """Fraction of zeros across the "real" weight matrices (ignores biases /
    LayerNorm, which are never pruned in practice)."""
    zeros, total = 0, 0
    for v in model.trainable_variables:
        if not any(k in v.name for k in kinds):
            continue
        arr = v.numpy()
        zeros += int((arr == 0).sum())
        total += int(arr.size)
    return zeros / max(total, 1)


# --------------------------------------------------------------------------
# 3. Latency / throughput benchmarking
# --------------------------------------------------------------------------


def measure_latency(predict_fn, inputs, warmup=5, runs=30, batch_size=1):
    """Run predict_fn(inputs) repeatedly and report wall-clock percentiles.

    Warm-up matters: the first calls pay for graph tracing, kernel autotuning
    and (on GPU) cuDNN algorithm selection. Reporting those numbers is the
    single most common benchmarking mistake in deployment work.
    """
    for _ in range(warmup):
        predict_fn(inputs)

    samples = []
    for _ in range(runs):
        t0 = time.perf_counter()
        predict_fn(inputs)
        samples.append((time.perf_counter() - t0) * 1000.0)

    samples = np.array(sorted(samples))
    p50 = float(np.percentile(samples, 50))
    return {
        "mean_ms": round(float(samples.mean()), 2),
        "p50_ms": round(p50, 2),
        "p90_ms": round(float(np.percentile(samples, 90)), 2),
        "p95_ms": round(float(np.percentile(samples, 95)), 2),
        "throughput_rps": round(batch_size / (p50 / 1000.0), 1),
    }


def device_label():
    return "GPU" if tf.config.list_physical_devices("GPU") else "CPU"


# --------------------------------------------------------------------------
# 4. The optimization ledger
# --------------------------------------------------------------------------


def _ledger_file():
    (_ROOT / "reports").mkdir(parents=True, exist_ok=True)
    return _ROOT / "reports" / "ledger.json"


def load_ledger():
    f = _ledger_file()
    if not f.exists():
        return []
    return json.loads(f.read_text())


def record(stage, **fields):
    """Insert or replace a ledger row. Stage names are unique keys, so
    re-running a cell updates the row instead of duplicating it."""
    ledger = [e for e in load_ledger() if e.get("stage") != stage]
    entry = {"stage": stage, "recorded_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    entry.update(fields)
    ledger.append(entry)
    _ledger_file().write_text(json.dumps(ledger, indent=2))
    return entry


def ledger_df(columns=None):
    import pandas as pd

    df = pd.DataFrame(load_ledger())
    if df.empty:
        return df
    preferred = [
        "stage",
        "model",
        "task",
        "dataset",
        "params_m",
        "size_mb",
        "quality",
        "quality_metric",
        "p50_ms",
        "p95_ms",
        "throughput_rps",
        "device",
        "notes",
    ]
    cols = columns or [c for c in preferred if c in df.columns]
    extra = [c for c in df.columns if c not in cols and c != "recorded_at"]
    return df[cols + extra]


# --------------------------------------------------------------------------
# 5. A small, explicit training loop
# --------------------------------------------------------------------------


def train(
    model,
    dataset,
    loss_fn,
    optimizer,
    epochs=1,
    steps_per_epoch=None,
    log_every=50,
    on_step_end=None,
    clip_norm=1.0,
):
    """Generic GradientTape loop.

    loss_fn(model, batch, training) -> scalar loss tensor.
    on_step_end(global_step) -> optional Python callback, used by the pruning
    lab to update sparsity masks between steps.
    """

    @tf.function
    def train_step(batch):
        with tf.GradientTape() as tape:
            loss = loss_fn(model, batch, True)
        grads = tape.gradient(loss, model.trainable_variables)
        pairs = [
            (g, v) for g, v in zip(grads, model.trainable_variables) if g is not None
        ]
        if clip_norm:
            gs, _ = tf.clip_by_global_norm([g for g, _ in pairs], clip_norm)
            pairs = list(zip(gs, [v for _, v in pairs]))
        optimizer.apply_gradients(pairs)
        return loss

    global_step = 0
    history = []
    for epoch in range(epochs):
        running, seen = 0.0, 0
        t0 = time.time()
        for step, batch in enumerate(dataset):
            loss = float(train_step(batch))
            running += loss
            seen += 1
            global_step += 1
            if on_step_end is not None:
                on_step_end(global_step)
            if log_every and global_step % log_every == 0:
                print(
                    f"  epoch {epoch + 1} | step {global_step:>5} | "
                    f"loss {running / seen:.4f}"
                )
                running, seen = 0.0, 0
            if steps_per_epoch and step + 1 >= steps_per_epoch:
                break
        history.append({"epoch": epoch + 1, "seconds": round(time.time() - t0, 1)})
        print(f"  epoch {epoch + 1} finished in {history[-1]['seconds']}s")
    return history


# --------------------------------------------------------------------------
# 6. Evaluation helpers
# --------------------------------------------------------------------------


def evaluate_accuracy(logits_fn, dataset):
    """logits_fn(features) -> array of shape [batch, num_classes]."""
    correct, total = 0, 0
    for features, labels in dataset:
        logits = np.asarray(logits_fn(features))
        preds = logits.argmax(axis=-1)
        labels = np.asarray(labels)
        correct += int((preds == labels).sum())
        total += int(labels.shape[0])
    return correct / max(total, 1)


def hf_logits_fn(model):
    """Wrap a Hugging Face TF model so it returns a plain logits tensor and is
    compiled once into a graph (fair, low-overhead benchmarking)."""

    @tf.function(reduce_retracing=True)
    def fn(features):
        return model(features, training=False).logits

    return fn


def banner(title):
    line = "=" * max(60, len(title) + 4)
    print(f"\n{line}\n  {title}\n{line}")
'''

KDKIT_SRC = r'''
"""kdkit.py - reusable knowledge-distillation utilities."""

import numpy as np
import tensorflow as tf

NEG_INF = -1e4


def soft_cross_entropy(teacher_logits, student_logits, T, mask=None):
    """Cross-entropy between temperature-softened teacher and student
    distributions. Optionally masks padded positions."""
    if mask is not None:
        penalty = (1.0 - tf.cast(mask, tf.float32)) * (-NEG_INF)
        teacher_logits = teacher_logits - penalty
        student_logits = student_logits - penalty
    t_probs = tf.nn.softmax(teacher_logits / T, axis=-1)
    s_logp = tf.nn.log_softmax(student_logits / T, axis=-1)
    return -tf.reduce_sum(t_probs * s_logp, axis=-1)


def make_classification_kd_loss(T=3.0, alpha=0.7):
    """Return loss_fn(model, batch, training) for single-label classification.

    Expected batch layout:
        (features, {"label": int32[B], "t_logits": float32[B, C]})
    """
    hard = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True, reduction=tf.keras.losses.Reduction.NONE
    )

    def loss_fn(model, batch, training):
        features, targets = batch
        logits = model(features, training=training).logits
        soft = soft_cross_entropy(targets["t_logits"], logits, T)
        hard_term = hard(targets["label"], logits)
        return tf.reduce_mean(alpha * (T**2) * soft + (1.0 - alpha) * hard_term)

    return loss_fn


def evenly_spaced_layers(n_teacher, n_student):
    """Teacher layer indices to copy into a student: evenly spaced, and always
    including the teacher's final layer."""
    step = n_teacher / n_student
    return [min(n_teacher - 1, int(round((i + 1) * step)) - 1) for i in range(n_student)]


def copy_encoder_weights(student, teacher, layer_map, encoder_attr="bert"):
    """Initialise a shallower student from a teacher of identical width."""
    s_main = getattr(student, encoder_attr)
    t_main = getattr(teacher, encoder_attr)
    s_main.embeddings.set_weights(t_main.embeddings.get_weights())
    for s_idx, t_idx in enumerate(layer_map):
        s_main.encoder.layer[s_idx].set_weights(t_main.encoder.layer[t_idx].get_weights())
    return layer_map


def cache_logits(model, features, batch_size=32, attr="logits"):
    """One teacher pass over a fixed dataset, for offline distillation."""
    n = features["input_ids"].shape[0]
    chunks = []
    for i in range(0, n, batch_size):
        batch = {k: tf.constant(v[i : i + batch_size]) for k, v in features.items()}
        chunks.append(getattr(model(batch, training=False), attr).numpy())
    return np.concatenate(chunks)
'''

from pathlib import Path
Path(ROOT, 'labkit.py').write_text(LABKIT_SRC)
Path(ROOT, 'kdkit.py').write_text(KDKIT_SRC)

import sys, importlib
sys.path.insert(0, ROOT)
import labkit as lk, kdkit
importlib.reload(lk); importlib.reload(kdkit)
lk.set_root(ROOT)
lk.banner('lab kit + kd kit ready')
print('ledger rows:', [r['stage'] for r in lk.load_ledger()])

---
## 1. Where we start

Bring the ledger forward and re-establish the baseline. Everything below is measured against the Lab 1 row.

In [ ]:
lk.ledger_df()

In [ ]:
from transformers import (AutoTokenizer, TFAutoModelForSequenceClassification,
                          TFBertForSequenceClassification, BertConfig)
from datasets import load_dataset

TEACHER_DIR = os.path.join(ROOT, "models", "teacher-bert-sst2")
if not os.path.isdir(TEACHER_DIR):
    raise FileNotFoundError(f"{TEACHER_DIR} not found - run Lab 1 Section 4 first.")

MAX_LEN, BATCH = 128, 32
N_TRAIN = 15_000

tokenizer = AutoTokenizer.from_pretrained(TEACHER_DIR)
teacher = TFAutoModelForSequenceClassification.from_pretrained(TEACHER_DIR)
_ = teacher(teacher.dummy_inputs, training=False)
teacher.trainable = False

sst2 = load_dataset("nyu-mll/glue", "sst2")
train_txt = sst2["train"].shuffle(seed=42).select(range(N_TRAIN))
val_txt = sst2["validation"]

def encode(texts, max_len=MAX_LEN, padding="max_length"):
    enc = tokenizer(list(texts), max_length=max_len, truncation=True,
                    padding=padding, return_tensors="np")
    return {k: np.asarray(v, np.int32) for k, v in enc.items()}

train_feats = encode(train_txt["sentence"])
train_labels = np.asarray(train_txt["label"], np.int32)
val_ds = (tf.data.Dataset.from_tensor_slices(
    (encode(val_txt["sentence"]), np.asarray(val_txt["label"], np.int32)))
    .batch(64).prefetch(tf.data.AUTOTUNE))

teacher_acc = lk.evaluate_accuracy(lk.hf_logits_fn(teacher), val_ds)
print(f"teacher: {teacher.num_parameters()/1e6:.1f}M params | "
      f"accuracy {teacher_acc:.4f}")

journey = [{"step": "0. baseline (BERT-base)", "params_m": round(teacher.num_parameters()/1e6, 1),
            "accuracy": round(teacher_acc, 4)}]

---
## 2. Step 1 - Distil

We reuse `kdkit` from Lab 2 unchanged. Only the loss head differs: single-label classification instead of span prediction, which is why `make_classification_kd_loss` was written to be task-agnostic.

Same recipe as Lab 2: evenly spaced layer initialisation, cached teacher logits, temperature 3, alpha 0.7.

**Expected time: 3–5 minutes.**

In [ ]:
STUDENT_LAYERS = 4
TEMPERATURE, ALPHA = 3.0, 0.7

lk.banner("caching teacher logits")
t_logits = kdkit.cache_logits(teacher, train_feats, batch_size=64)
print("cached", t_logits.shape)

layer_map = kdkit.evenly_spaced_layers(teacher.config.num_hidden_layers, STUDENT_LAYERS)
student_cfg = BertConfig.from_dict(teacher.config.to_dict())
student_cfg.num_hidden_layers = STUDENT_LAYERS

student = TFBertForSequenceClassification(student_cfg)
_ = student(teacher.dummy_inputs, training=False)
kdkit.copy_encoder_weights(student, teacher, layer_map)
student.classifier.set_weights(teacher.classifier.get_weights())
student.bert.pooler.set_weights(teacher.bert.pooler.get_weights())
print("student initialised from teacher layers", layer_map)
print(f"student: {student.num_parameters()/1e6:.1f}M params "
      f"({student.num_parameters()/teacher.num_parameters():.0%} of teacher)")

In [ ]:
kd_ds = (tf.data.Dataset.from_tensor_slices(
            (train_feats, {"label": train_labels, "t_logits": t_logits}))
         .shuffle(4096, seed=42).batch(BATCH).prefetch(tf.data.AUTOTUNE))

EPOCHS = 2
steps = int(np.ceil(len(train_labels) / BATCH)) * EPOCHS
sched = tf.keras.optimizers.schedules.PolynomialDecay(5e-5, steps, end_learning_rate=0.0)
opt = tf.keras.optimizers.Adam(learning_rate=sched)

lk.banner(f"distilling to {STUDENT_LAYERS} layers | {steps} steps")
lk.train(student, kd_ds, kdkit.make_classification_kd_loss(TEMPERATURE, ALPHA),
         opt, epochs=EPOCHS, log_every=200)

distilled_acc = lk.evaluate_accuracy(lk.hf_logits_fn(student), val_ds)
print(f"\ndistilled accuracy {distilled_acc:.4f} (teacher {teacher_acc:.4f}, "
      f"delta {distilled_acc - teacher_acc:+.4f})")
journey.append({"step": "1. + distilled to 4L",
                "params_m": round(student.num_parameters()/1e6, 1),
                "accuracy": round(distilled_acc, 4)})

---
## 3. Step 2 - Structured pruning on the distilled student

The interesting question: **is there redundancy left?** Distillation already compressed the model's function into fewer layers, so it is entirely plausible that the surviving feed-forward neurons are all doing real work.

We prune less aggressively than in Lab 4 - 3072 → 2048 rather than → 1536 - precisely because we expect less slack. Measure, then decide.

**Expected time: 2–3 minutes.**

In [ ]:
NEW_INTERMEDIATE = 2048

slim_cfg = BertConfig.from_dict(student.config.to_dict())
slim_cfg.intermediate_size = NEW_INTERMEDIATE
slim = TFBertForSequenceClassification(slim_cfg)
_ = slim(teacher.dummy_inputs, training=False)

slim.bert.embeddings.set_weights(student.bert.embeddings.get_weights())
for src, dst in zip(student.bert.encoder.layer, slim.bert.encoder.layer):
    dst.attention.set_weights(src.attention.get_weights())

    w_in, b_in = src.intermediate.dense.get_weights()
    w_out, b_out = src.bert_output.dense.get_weights()
    importance = np.linalg.norm(w_in, axis=0) * np.linalg.norm(w_out, axis=1)
    keep = np.sort(np.argsort(importance)[-NEW_INTERMEDIATE:])

    dst.intermediate.dense.set_weights([w_in[:, keep], b_in[keep]])
    dst.bert_output.dense.set_weights([w_out[keep, :], b_out])
    dst.bert_output.LayerNorm.set_weights(src.bert_output.LayerNorm.get_weights())

slim.bert.pooler.set_weights(student.bert.pooler.get_weights())
slim.classifier.set_weights(student.classifier.get_weights())

print(f"before surgery accuracy: {distilled_acc:.4f}")
print(f"after  surgery accuracy: "
      f"{lk.evaluate_accuracy(lk.hf_logits_fn(slim), val_ds):.4f}")
print(f"parameters {student.num_parameters()/1e6:.1f}M -> {slim.num_parameters()/1e6:.1f}M")

In [ ]:
RECOVER_STEPS = 500
scce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

def clf_loss(model, batch, training):
    features, labels = batch
    return scce(labels, model(features, training=training).logits)

recover_ds = (tf.data.Dataset.from_tensor_slices((train_feats, train_labels))
              .shuffle(4096, seed=42).batch(BATCH).prefetch(tf.data.AUTOTUNE).repeat())

sched = tf.keras.optimizers.schedules.PolynomialDecay(3e-5, RECOVER_STEPS, end_learning_rate=0.0)
lk.banner("recovering the pruned student")
lk.train(slim, recover_ds, clf_loss, tf.keras.optimizers.Adam(learning_rate=sched),
         epochs=1, steps_per_epoch=RECOVER_STEPS, log_every=150)

pruned_acc = lk.evaluate_accuracy(lk.hf_logits_fn(slim), val_ds)
print(f"\npruned + recovered accuracy {pruned_acc:.4f} "
      f"(distilled was {distilled_acc:.4f}, delta {pruned_acc - distilled_acc:+.4f})")
journey.append({"step": "2. + FFN pruned to 2048",
                "params_m": round(slim.num_parameters()/1e6, 1),
                "accuracy": round(pruned_acc, 4)})
pd.DataFrame(journey)

**Read the delta carefully.** If pruning cost you noticeably more here than the equivalent step cost on the dense teacher in Lab 4, you have measured the levers *competing* - distillation already consumed the redundancy pruning wanted. If it cost about the same, they compound.

Either result is a legitimate finding and both are worth reporting. What is not legitimate is assuming compounding without checking, which is how optimization programmes end up shipping a model that is 3× smaller and 4 points worse than anyone expected.

---
## 4. Step 3 - Quantize

We apply the Lab 3 method: per-channel symmetric int8 on 2-D kernels, in simulation, to establish the accuracy cost. The size claim comes from format arithmetic, as it did in Lab 3, and the real int8 execution happens in the serving runtime on Day 2.

In [ ]:
def quantize_dequantize(x, num_bits=8, axis=None):
    x = tf.convert_to_tensor(x, tf.float32)
    qmax = 2 ** (num_bits - 1) - 1
    reduce_axes = None if axis is None else [i for i in range(len(x.shape)) if i != axis]
    scale = tf.maximum(tf.reduce_max(tf.abs(x), axis=reduce_axes, keepdims=True) / qmax, 1e-12)
    return scale * tf.clip_by_value(tf.round(x / scale), -qmax - 1, qmax)

FP32_WEIGHTS = {v.name: v.numpy().copy() for v in slim.weights}

def restore_fp32():
    for v in slim.weights:
        v.assign(FP32_WEIGHTS[v.name])

def apply_int8(include_embeddings=False):
    restore_fp32()
    for v in slim.weights:
        name = v.name.lower()
        if "kernel" in name and len(v.shape) == 2:
            v.assign(quantize_dequantize(v, axis=1))
        elif include_embeddings and "embeddings" in name and len(v.shape) == 2:
            v.assign(quantize_dequantize(v, axis=None))

apply_int8(include_embeddings=False)
q_acc = lk.evaluate_accuracy(lk.hf_logits_fn(slim), val_ds)
apply_int8(include_embeddings=True)
q_acc_emb = lk.evaluate_accuracy(lk.hf_logits_fn(slim), val_ds)
restore_fp32()

print(f"fp32                     {pruned_acc:.4f}")
print(f"int8 kernels             {q_acc:.4f}  ({q_acc - pruned_acc:+.4f})")
print(f"int8 kernels + embeddings {q_acc_emb:.4f}  ({q_acc_emb - pruned_acc:+.4f})")

kernels = [v for v in slim.weights if "kernel" in v.name.lower() and len(v.shape) == 2]
embeds  = [v for v in slim.weights if "embeddings" in v.name.lower() and len(v.shape) == 2]
_grouped = {v.name for v in kernels} | {v.name for v in embeds}
others  = [v for v in slim.weights if v.name not in _grouped]
p = lambda vs: sum(int(np.prod(v.shape)) for v in vs)
fp32_mb = (p(kernels) + p(embeds) + p(others)) * 4 / 1e6
int8_mb = (p(kernels) * 1 + p(embeds) * 4 + p(others) * 4) / 1e6
int8e_mb = (p(kernels) * 1 + p(embeds) * 1 + p(others) * 4) / 1e6
print(f"\nprojected size: fp32 {fp32_mb:.1f} MB | int8 kernels {int8_mb:.1f} MB | "
      f"int8 + embeddings {int8e_mb:.1f} MB")

journey.append({"step": "3. + int8 kernels",
                "params_m": round(slim.num_parameters()/1e6, 1),
                "accuracy": round(q_acc, 4)})

Note where the size now sits. After distillation the embedding table dominates the parameter count, so **quantizing embeddings is a much larger relative win on the compressed model than it was on BERT-base.** The order of operations changed which decision matters - a good illustration of why you re-measure at each step rather than reusing conclusions from an earlier configuration.

We keep embeddings at fp32 for the deployed artifact, on the grounds that the extra accuracy cost is not worth the size saving at this scale. Record the alternative in the manifest so the decision is visible to whoever revisits it.

---
## 5. Step 4 - Smaller inputs

The case study's most counter-intuitive finding: batching required padding every input to a common length, and removing that padding by serving at batch size 1 was faster overall.

Our SST-2 inputs have a median of about 9 tokens. We have been padding every one to 128. Let us price that.

In [ ]:
lengths = [len(tokenizer(s)["input_ids"]) for s in val_txt["sentence"]]
print(f"true token length: median {np.median(lengths):.0f} | p95 "
      f"{np.percentile(lengths, 95):.0f} | max {max(lengths)}")

restore_fp32()
fn = lk.hf_logits_fn(slim)
rows = []
for L in (16, 32, 64, 128):
    feats = encode(val_txt["sentence"][:64], max_len=L)
    lat = lk.measure_latency(fn, {k: tf.constant(v[:1]) for k, v in feats.items()},
                             warmup=10, runs=60, batch_size=1)
    rows.append({"seq_len": L, "p50_ms": lat["p50_ms"], "p95_ms": lat["p95_ms"]})
shapes = pd.DataFrame(rows)
shapes["vs_128"] = (shapes.p50_ms.iloc[-1] / shapes.p50_ms).round(2)
shapes

Each row is the same model doing the same job on the same text - only the padding differs. The speedup in the last column is free: no quality cost at all, because the padded positions were masked out and contributed nothing to the output in the first place.

There is a catch, and it is the reason many teams never collect this win: **variable input shapes cause graph retracing.** Every new sequence length triggers a fresh trace, and the first request at each length pays for it. The standard mitigation is **bucketing** - round every request up to the nearest of a handful of lengths (say 16, 32, 64, 128), so you get most of the saving with a bounded number of traced graphs. That is a Day 2 serving-configuration decision, and it belongs in the manifest.

---
## 6. Measure on the target hardware

Every number so far came from a T4. Day 2 deploys to **AWS ECS**, and the case study's conclusion was that CPU serving becomes viable once the model is small enough. So we re-measure on CPU - a Colab CPU is not a Xeon, but the *ratio* between the baseline and the optimized model transfers, and that ratio is what drives the deployment decision.

**Expected time: 2–3 minutes** - the fp32 BERT-base baseline on CPU is slow, which is exactly the point.

One caveat to state honestly: the model variables were created on the GPU, so forcing the ops onto CPU adds a device transfer per call that a real CPU-only host would not pay. The absolute milliseconds are therefore pessimistic. The **ratio** between baseline and optimized is what we use, and that survives the caveat - both configurations pay the same overhead.

In [ ]:
def cpu_latency(model, seq_len, runs=20):
    feats = encode(val_txt["sentence"][:8], max_len=seq_len)
    with tf.device("/CPU:0"):
        cpu_fn = tf.function(lambda x: model(x, training=False).logits,
                             reduce_retracing=True)
        sample = {k: tf.constant(v[:1]) for k, v in feats.items()}
        return lk.measure_latency(cpu_fn, sample, warmup=3, runs=runs, batch_size=1)

lk.banner("CPU latency, batch size 1")
cpu_rows = []
for label, model_ref, L in [
    ("baseline BERT-base @128", teacher, 128),
    ("optimized 4L+FFN  @128", slim, 128),
    ("optimized 4L+FFN  @32",  slim, 32),
    ("optimized 4L+FFN  @16",  slim, 16),
]:
    lat = cpu_latency(model_ref, L)
    cpu_rows.append({"config": label, **lat})
    print(f"  {label:<26} p50 {lat['p50_ms']:8.1f} ms  p95 {lat['p95_ms']:8.1f} ms")

cpu_df = pd.DataFrame(cpu_rows)
cpu_df["speedup_vs_baseline"] = (cpu_df.p50_ms.iloc[0] / cpu_df.p50_ms).round(2)
cpu_df

In [ ]:
# Feed the measured CPU numbers back into the Lab 1 cost model.
DAILY_REQUESTS, PEAK_FACTOR, TARGET_UTIL, HOURS = 1_000_000_000, 2.5, 0.60, 730
GPU_PRICE, CPU_PRICE = 1.006, 0.340      # illustrative us-east-1 on-demand rates
GPU_STREAMS, CPU_STREAMS = 1, 8

def monthly(p50_ms, price, streams):
    peak_qps = DAILY_REQUESTS / 86_400 * PEAK_FACTOR
    per_instance = (1000.0 / p50_ms) * streams
    n = int(np.ceil(peak_qps / (per_instance * TARGET_UTIL)))
    return n, round(n * price * HOURS)

base_n, base_cost = monthly(cpu_df.p50_ms.iloc[0], CPU_PRICE, CPU_STREAMS)
opt_n, opt_cost = monthly(cpu_df.p50_ms.iloc[-1], CPU_PRICE, CPU_STREAMS)
print(f"baseline on CPU : {base_n:>7,} instances -> ${base_cost:>12,}/month")
print(f"optimized on CPU: {opt_n:>7,} instances -> ${opt_cost:>12,}/month")
print(f"\nreduction: {base_cost/max(opt_cost,1):.1f}x  "
      f"(${base_cost - opt_cost:,}/month at this traffic level)")
print("\nRevisit your Lab 1 Section 2.5 answers against these numbers.")

---
## 7. Export the deployment artifact

Now we produce what Day 2 consumes. Three decisions are baked into the export and each has a consequence downstream:

1. **Dynamic sequence length** (`[None, None]` in the signature) so the serving stack can use bucketing rather than always padding to 128.
2. **Dynamic batch size** so the same graph serves interactive and batch traffic.
3. **Tokenization stays outside the graph.** TensorFlow Serving will receive token IDs; the Flask layer owns tokenization. That keeps the SavedModel portable and the tokenizer independently versionable - at the cost of requiring the two to be deployed as a matched pair, which the manifest records.

In [ ]:
EXPORT = os.path.join(ROOT, "models", "day2-deploy")
SAVED_MODEL_DIR = os.path.join(EXPORT, "saved_model", "1")   # TF Serving wants a version dir
os.makedirs(EXPORT, exist_ok=True)

restore_fp32()

@tf.function(input_signature=[{
    "input_ids":      tf.TensorSpec([None, None], tf.int32, name="input_ids"),
    "attention_mask": tf.TensorSpec([None, None], tf.int32, name="attention_mask"),
    "token_type_ids": tf.TensorSpec([None, None], tf.int32, name="token_type_ids"),
}])
def serving_fn(inputs):
    logits = slim(inputs, training=False).logits
    probs = tf.nn.softmax(logits, axis=-1)
    return {"logits": logits,
            "probabilities": probs,
            "predicted_class": tf.argmax(probs, axis=-1, output_type=tf.int32)}

tf.saved_model.save(slim, SAVED_MODEL_DIR, signatures={"serving_default": serving_fn})
tokenizer.save_pretrained(os.path.join(EXPORT, "tokenizer"))
slim.save_pretrained(os.path.join(EXPORT, "hf_model"))

print("SavedModel :", f"{lk.size_mb(SAVED_MODEL_DIR):.1f} MB")
print("tokenizer  :", f"{lk.size_mb(os.path.join(EXPORT, 'tokenizer')):.2f} MB")

In [ ]:
!saved_model_cli show --dir "{SAVED_MODEL_DIR}" --tag_set serve --signature_def serving_default

In [ ]:
# Validate exactly as a deployment engineer would: load the artifact fresh,
# ignore the in-memory model, and check it behaves at several shapes.
loaded = tf.saved_model.load(SAVED_MODEL_DIR)
infer = loaded.signatures["serving_default"]

samples = ["a genuinely delightful film",
           "dull, overlong and utterly forgettable",
           "the performances carry an otherwise thin script"]

for max_len in (16, 64):
    enc = tokenizer(samples, max_length=max_len, truncation=True,
                    padding="max_length", return_tensors="tf")
    out = infer(input_ids=tf.cast(enc["input_ids"], tf.int32),
                attention_mask=tf.cast(enc["attention_mask"], tf.int32),
                token_type_ids=tf.cast(enc["token_type_ids"], tf.int32))
    labels = ["negative", "positive"]
    print(f"\nseq_len={max_len}")
    for text, cls, prob in zip(samples, out["predicted_class"].numpy(),
                               out["probabilities"].numpy()):
        print(f"  {labels[cls]:<9} p={prob[cls]:.3f}  |  {text}")

### 7.1 The deployment manifest

An artifact without a manifest is a liability. This file tells the Day 2 team what the model expects, what it was measured at, and which decisions are open for revisiting - so nobody has to reverse-engineer any of it from the graph.

In [ ]:
manifest = {
    "model_name": "sst2-sentiment-optimized",
    "version": 1,
    "created": time.strftime("%Y-%m-%d"),
    "task": "binary sentiment classification",
    "labels": {"0": "negative", "1": "positive"},

    "architecture": {
        "base": "bert-base-uncased",
        "hidden_size": int(slim.config.hidden_size),
        "num_hidden_layers": int(slim.config.num_hidden_layers),
        "intermediate_size": int(slim.config.intermediate_size),
        "parameters_m": round(slim.num_parameters() / 1e6, 2),
    },

    "optimizations_applied": [
        {"technique": "knowledge distillation",
         "detail": f"12 -> {STUDENT_LAYERS} layers, teacher layers {layer_map}, "
                   f"T={TEMPERATURE}, alpha={ALPHA}"},
        {"technique": "structured pruning",
         "detail": f"feed-forward {teacher.config.intermediate_size} -> {NEW_INTERMEDIATE}, "
                   f"norm-product importance, {RECOVER_STEPS} recovery steps"},
        {"technique": "quantization",
         "detail": "int8 per-channel symmetric on 2-D kernels; validated in simulation, "
                   "apply in the serving runtime"},
    ],

    "serving": {
        "format": "SavedModel",
        "signature": "serving_default",
        "inputs": ["input_ids", "attention_mask", "token_type_ids"],
        "input_dtype": "int32",
        "input_shape": "[batch, seq_len] - both dynamic",
        "outputs": ["logits", "probabilities", "predicted_class"],
        "tokenizer": "bert-base-uncased WordPiece, bundled under tokenizer/",
        "tokenization_owner": "application layer (Flask), not the SavedModel",
        "recommended_seq_buckets": [16, 32, 64, 128],
        "max_seq_len": MAX_LEN,
    },

    "measured": {
        "accuracy_sst2_fp32": round(pruned_acc, 4),
        "accuracy_sst2_int8_simulated": round(q_acc, 4),
        "teacher_accuracy": round(teacher_acc, 4),
        "quality_retained_vs_teacher": round(pruned_acc / teacher_acc, 4),
        "gpu_p50_ms_seq128_batch1": float(shapes.p50_ms.iloc[-1]),
        "cpu_p50_ms_seq128_batch1": float(cpu_df.p50_ms.iloc[1]),
        "cpu_p50_ms_seq16_batch1": float(cpu_df.p50_ms.iloc[-1]),
        "cpu_speedup_vs_baseline": float(cpu_df.speedup_vs_baseline.iloc[-1]),
        "savedmodel_size_mb": round(lk.size_mb(SAVED_MODEL_DIR), 1),
        "projected_int8_size_mb": round(int8_mb, 1),
    },

    "open_decisions": [
        "Embedding table left at fp32. Quantizing it would save roughly "
        f"{int8_mb - int8e_mb:.0f} MB for about "
        f"{abs(q_acc_emb - q_acc)*100:.2f} accuracy points - revisit if memory-bound.",
        "Sequence bucketing is recommended but not enforced by the signature; "
        "the serving layer must implement it.",
        "Batching strategy unresolved: measure batch-1 against small batches on the "
        "actual ECS instance type before fixing it.",
    ],

    "day2_checklist": [
        "Serve saved_model/1 with TensorFlow Serving; confirm the model loads and "
        "the signature is discovered.",
        "Build a Flask API that tokenizes, buckets the sequence length, calls TF Serving "
        "and maps class indices to labels.",
        "Containerise both with Docker; keep the tokenizer in the image alongside the model.",
        "Push to ECR and deploy to ECS; size tasks against the CPU latency in 'measured'.",
        "Add a health check that runs one known input and asserts the expected label.",
    ],
}

manifest_path = os.path.join(EXPORT, "deployment_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2)[:1600], "\n...")
print("\nwritten to", manifest_path)

In [ ]:
# A reference preprocessing module so the Day 2 Flask layer starts from
# something known-correct rather than a reimplementation.
PREPROCESS = r'''
# preprocess.py - request-side preparation for sst2-sentiment-optimized.
#
# The SavedModel accepts token IDs only. Everything between raw text and those
# IDs lives here, and must be deployed as a matched pair with the model version.

import numpy as np
from transformers import AutoTokenizer

BUCKETS = [16, 32, 64, 128]
LABELS = {0: "negative", 1: "positive"}


class Preprocessor:
    def __init__(self, tokenizer_dir):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)

    def bucket_for(self, texts):
        longest = max(len(self.tokenizer(t)["input_ids"]) for t in texts)
        for b in BUCKETS:
            if longest <= b:
                return b
        return BUCKETS[-1]

    def encode(self, texts):
        # Pad to the smallest bucket that fits: keeps the number of traced
        # graphs bounded while avoiding worst-case padding on every request.
        length = self.bucket_for(texts)
        enc = self.tokenizer(texts, max_length=length, truncation=True,
                             padding="max_length", return_tensors="np")
        return {k: np.asarray(v, dtype=np.int32) for k, v in enc.items()}

    @staticmethod
    def decode(probabilities):
        out = []
        for row in np.asarray(probabilities):
            idx = int(np.argmax(row))
            out.append({"label": LABELS[idx], "confidence": float(row[idx])})
        return out
'''

with open(os.path.join(EXPORT, "preprocess.py"), "w") as f:
    f.write(PREPROCESS)

print("export contents:")
for path in sorted(Path(EXPORT).rglob("*")):
    if path.is_file():
        print(f"  {str(path.relative_to(EXPORT)):<48} {lk.size_mb(path):8.2f} MB")
print(f"\ntotal: {lk.size_mb(EXPORT):.1f} MB")

---
## 8. The full picture

In [ ]:
lk.record("capstone-optimized", model=f"BERT {STUDENT_LAYERS}L, FFN {NEW_INTERMEDIATE}, int8-ready",
          task="binary sentiment", dataset="SST-2",
          params_m=round(slim.num_parameters()/1e6, 1),
          size_mb=round(int8_mb, 1),
          quality=round(q_acc, 4), quality_metric="accuracy",
          p50_ms=float(shapes.p50_ms.iloc[-1]),
          p95_ms=float(shapes.p95_ms.iloc[-1]),
          throughput_rps=round(1000.0 / float(shapes.p50_ms.iloc[-1]), 1),
          device=lk.device_label(),
          cpu_p50_ms=float(cpu_df.p50_ms.iloc[-1]),
          notes="distil + structured prune + int8; dynamic shapes; Day 2 artifact")

journey.append({"step": "4. + dynamic shapes (seq 16)",
                "params_m": round(slim.num_parameters()/1e6, 1),
                "accuracy": round(q_acc, 4)})
lk.ledger_df()

In [ ]:
import matplotlib.pyplot as plt

j = pd.DataFrame(journey)
teacher_size = teacher.num_parameters() * 4 / 1e6
sizes = [teacher_size, student.num_parameters()*4/1e6, fp32_mb, int8_mb, int8_mb]
cpu_p50 = [cpu_df.p50_ms.iloc[0], None, cpu_df.p50_ms.iloc[1], cpu_df.p50_ms.iloc[1],
           cpu_df.p50_ms.iloc[-1]]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
labels = [s.split(". ", 1)[-1] for s in j.step]

axes[0].bar(range(len(sizes)), sizes, color="#4C72B0")
axes[0].set_title("Model size (MB)"); axes[0].set_ylabel("MB")

axes[1].plot(range(len(j)), j.accuracy, marker="o", color="#DD8452")
axes[1].axhline(teacher_acc, ls="--", c="grey", label="teacher")
axes[1].set_title("SST-2 accuracy"); axes[1].legend()

pts = [(i, v) for i, v in enumerate(cpu_p50) if v is not None]
axes[2].plot([p[0] for p in pts], [p[1] for p in pts], marker="s", color="#55A868")
axes[2].set_title("CPU p50 latency (ms), batch 1"); axes[2].set_yscale("log")

for ax in axes:
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    ax.grid(alpha=0.3)
plt.suptitle("Day 1 optimization journey", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, "reports", "day1_journey.png"), dpi=120,
            bbox_inches="tight")
plt.show()

In [ ]:
report_path = os.path.join(ROOT, "reports", "day1_summary.md")
df = lk.ledger_df()
with open(report_path, "w") as f:
    f.write("# Day 1 - LLM optimization results\n\n")
    f.write(df.to_markdown(index=False))
    f.write("\n\n## Optimization journey\n\n")
    f.write(pd.DataFrame(journey).to_markdown(index=False))
    f.write("\n\n## CPU latency\n\n")
    f.write(cpu_df.to_markdown(index=False))
    f.write("\n\n## Sequence-length sensitivity (GPU)\n\n")
    f.write(shapes.to_markdown(index=False))
print("summary written to", report_path)
print(df.to_string(index=False))

---
## 9. Wrap-up and handoff

### What Day 1 delivered

Starting from a fine-tuned BERT-base classifier, you applied the same three levers the case study used, in the same order, and measured after each:

- **Distillation** removed two-thirds of the encoder layers.
- **Structured pruning** removed a third of the feed-forward width from what remained - and, unlike the unstructured pruning in Lab 4, it made the model genuinely faster.
- **Quantization** cut the weight footprint roughly fourfold for a fraction of a point of accuracy.
- **Dynamic shapes** removed padding waste for free.

The end state is not just "a faster model". It is a model in a **different cost class** - one whose latency on a CPU is in the range where CPU serving is the obvious choice rather than a compromise. That is the same conclusion the case study reached, arrived at from your own measurements.

### The methodological points that matter more than the numbers

1. **Measure after every step.** Compounding is a hypothesis until you test it.
2. **Be explicit about what each measurement establishes.** Simulated quantization proves quality survives; a runtime proves size and speed. Never present one as the other.
3. **Benchmark on the target hardware.** The GPU numbers led to one conclusion; the CPU numbers led to a much stronger one.
4. **Record the decisions, not only the results.** The manifest's `open_decisions` section is what makes this artifact maintainable by someone who was not in the room.

### The handoff to Day 2

Everything the deployment work needs is under `models/day2-deploy/`:

```
day2-deploy/
├── saved_model/1/            TF Serving loads this directly
├── tokenizer/                must ship with the model, matched by version
├── hf_model/                 for further training or re-export
├── preprocess.py             bucketing + label mapping for the Flask layer
└── deployment_manifest.json  contract, measurements, open decisions
```

**Day 2 will:**

1. Serve `saved_model/1` with **TensorFlow Serving**.
2. Wrap it in a **Flask API** that owns tokenization, sequence bucketing and label mapping.
3. Package both into **Docker** images.
4. Deploy to **AWS ECS**, sizing tasks against the CPU latency recorded in the manifest.
5. Apply the cost strategies - right-sizing, autoscaling, Spot capacity, caching - against the cost model you have now built twice with real numbers.

### Final checkpoint

- [ ] `models/day2-deploy/` contains all five items above.
- [ ] `saved_model_cli` shows a `serving_default` signature with dynamic batch and sequence dimensions.
- [ ] The reloaded model classifies correctly at more than one sequence length.
- [ ] `reports/day1_summary.md` and `reports/day1_journey.png` exist.
